In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import binomtest, spearmanr, fisher_exact, hypergeom, chi2
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

In [4]:
COLS = {"gene": "primerid", "fdr": "fdr", "logfc": "logFC"}
# Analysis mode:
#   'sig_both'    -> only genes significant at BOTH timepoints (strict) - for fisher †est and overlaps
#   'all_tested'  -> all common genes (uses raw logFC for rho + sign test) - for computing correlation of logFCs
MODE = "sig_both"

In [24]:
#### al combined ##

In [8]:
# loop over all celltypes acorss all treatments in a function#

CELLTYPES = ["Monocytes","NK","pDCs","ASCs","B_cells","CD2_neg_T","CD2_pos_T","CD4_Tcells","CD8T_cells"]
#list contrats specified as in DE list from MAST
CONTRASTS = {
    "EP": "ext_per",
    "EC": "ext_con",
    "PC": "per_con",
}
#then define fragments are celltype+contrasts
#base file paths
BASE14 = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/Project_Fang_10X/Project_Fang_10X/MAST_DE_14dpi")
BASE84 = Path("/work/ABG/mkapoor/mkapoor/PRRSV/PRRSV_cellranger_v97/MAST_DE_84dpi")
#output paths
OUTROOT = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84")
OUTROOT.mkdir(parents=True, exist_ok=True)
#outdir = OUTROOT / contrast / celltype
#set thresholds for easier control
FDR_CUTOFF = 0.05
LFC_CUTOFF = 0.05
MIN_LIST_SIZE_FOR_TESTS = 50
COLS = {"gene": "primerid", "fdr": "fdr", "logfc": "logFC"}
MODE = "sig_both"  # and "all_tested"
UNIVERSE_MODE = "intersection" 

#function to loadd all files and define thresholds
#need to define another function because some of the colums might  be case sensitive(pick)
def load_and_standardize(fn, cols):
    df = pd.read_csv(fn)
    def pick(cands):
        low = {c.lower(): c for c in df.columns}
        for c in cands:
            if c.lower() in low:
                return low[c.lower()]
        raise KeyError(f"None of {cands} in {list(df.columns)}")
        #for each column you descrive pick the reapective one out of otehrs
    gene_col = pick([cols["gene"], "gene", "primerid", "symbol"])
    fdr_col  = pick([cols["fdr"], "fdr", "adj.P.Val", "padj", "qvalue"])
    lfc_col  = pick([cols["logfc"], "logfc", "log2fc", "avg_log2FC", "coef"])
    #for simplicity - rename columns and remove all duplicates after first instance
    df = df[[gene_col, fdr_col, lfc_col]].rename(columns={gene_col:"gene", fdr_col:"fdr", lfc_col:"logFC"})
    df["fdr"]   = pd.to_numeric(df["fdr"], errors="coerce")
    #df["gene"]   = pd.to_numeric(df["gene"], errors="coerce")
    df["logFC"] = pd.to_numeric(df["logFC"], errors="coerce")
    df = df.dropna(subset=["gene","fdr","logFC"])
    # keep min FDR; and by largest |logFC|
    df["_abs"] = df["logFC"].abs()
    df = (df.sort_values(["gene","fdr","_abs"], ascending=[True, True, False])
            .groupby("gene", as_index=False).first()
            .drop(columns=["_abs"]))
    #define up and down lists and gice them name acc to direction and |0.05| + FDR threshold
    cond_up   = (df["fdr"] < FDR_CUTOFF) & (df["logFC"] >  LFC_CUTOFF)
    cond_down = (df["fdr"] < FDR_CUTOFF) & (df["logFC"] < -LFC_CUTOFF)
    df["direction"] = np.where(cond_up, "Upregulated",
                        np.where(cond_down, "Downregulated", "Non-significant"))
    return df
#Method 1: to find similairity measure between DEG list do jaccard similarity formula 
def jaccard(a, b):
    a, b = set(a), set(b)
    return np.nan if (not a and not b) else len(a & b) / len(a | b)
#fragmnets are needed here to call files from different base locations
def read_files(base_dir, celltype, frag):
    ct_dir = base_dir / celltype
    candidates = [
        ct_dir / f"Mast_{celltype}_{frag}.csv"
    ]
    for p in candidates:
        if p.exists():
            return p
    if ct_dir.exists():
        for p in ct_dir.glob("*.csv"):
            if frag in p.name: return p
    raise FileNotFoundError(f"No MAST file for {celltype} + '{frag}' in {ct_dir}")
#Method 2: Do fisher test for meta nalaysis using chi-square tests for independent p value lists of DEGs at differneet timepoints
def fisher_method_p(pvals):
    pvals = [p for p in pvals if (p is not None and not np.isnan(p))]
    if len(pvals) == 0: return np.nan
    stat = -2 * np.sum(np.log(pvals))
    df = 2 * len(pvals)
    return chi2.sf(stat, df)

#Method 3: Do fisher one sided overlap test for testing concordance between DEGs UpUP and DownDown
#create 2x2 contingency table where N= total number of genes in universe (defined in anlayze function)
def fisher_overlap(a, b, k, N, alternative="greater"):
    table = np.array([[k,       a - k],
                      [b - k,   N - a - b + k]])
    # (_ will return oddsratio, pvalue)
    _, p = fisher_exact(table, alternative=alternative)
    return p

#this will set signs and define universe. then compute directuon baased test, jaccard , fihsher overlap of concordance, flipped, overlal_fisher, expects
# all, jaccard all and fold enrichment all 
def analyze(celltype, contrast_key):
    #set fragemnt
    frag = CONTRASTS[contrast_key]
    fn14 = read_files(BASE14, celltype, frag)
    fn84 = read_files(BASE84, celltype, frag)
    # load files with names
    d14 = load_and_standardize(fn14, COLS).add_suffix("_14").rename(columns={"gene_14":"gene"})
    d84 = load_and_standardize(fn84, COLS).add_suffix("_84").rename(columns={"gene_84":"gene"})
    def is_sig(df):
        return (df["fdr_14"] < FDR_CUTOFF) & (df["logFC_14"].abs() > LFC_CUTOFF) if "fdr_14" in df else \
        (df["fdr_84"] < FDR_CUTOFF) & (df["logFC_84"].abs() > LFC_CUTOFF)
    #print raw 
    d14_raw = d14.rename(columns={"fdr_14":"fdr","logFC_14":"logFC"})
    d84_raw = d84.rename(columns={"fdr_84":"fdr","logFC_84":"logFC"})

    DE14_raw_genes = set(d14_raw.loc[(d14_raw["fdr"] < FDR_CUTOFF) & (d14_raw["logFC"].abs() > LFC_CUTOFF), "gene"])
    DE84_raw_genes = set(d84_raw.loc[(d84_raw["fdr"] < FDR_CUTOFF) & (d84_raw["logFC"].abs() > LFC_CUTOFF), "gene"])
    n_DE14_raw = len(DE14_raw_genes)
    n_DE84_raw = len(DE84_raw_genes)
    genes14 = set(d14["gene"]); genes84 = set(d84["gene"])
    if UNIVERSE_MODE == "intersection":
        universe = sorted(genes14 & genes84)
    elif UNIVERSE_MODE == "union":
        universe = sorted(genes14 | genes84)
    else:
        raise ValueError("intersection or union?")

    d14u = d14[d14["gene"].isin(universe)].set_index("gene")
    d84u = d84[d84["gene"].isin(universe)].set_index("gene")
    merged = d14u.join(d84u, how="outer") 
    sig14 = merged["fdr_14"].lt(FDR_CUTOFF) & (merged["logFC_14"].abs().gt(LFC_CUTOFF))
    sig84 = merged["fdr_84"].lt(FDR_CUTOFF) & (merged["logFC_84"].abs().gt(LFC_CUTOFF))
    DE14_inter = set(merged.index[sig14])
    DE84_inter = set(merged.index[sig84])
    n_DE14_inter = len(DE14_inter)
    n_DE84_inter = len(DE84_inter)
    base = merged[sig14 & sig84].copy() if MODE == "sig_both" else merged.copy()
    # Direction sets
    up14    = set(merged.index[(merged["fdr_14"]<FDR_CUTOFF) & (merged["logFC_14"]> LFC_CUTOFF)])
    down14  = set(merged.index[(merged["fdr_14"]<FDR_CUTOFF) & (merged["logFC_14"]<-LFC_CUTOFF)])
    up84    = set(merged.index[(merged["fdr_84"]<FDR_CUTOFF) & (merged["logFC_84"]> LFC_CUTOFF)])
    down84  = set(merged.index[(merged["fdr_84"]<FDR_CUTOFF) & (merged["logFC_84"]<-LFC_CUTOFF)])
    #unique genes
    up14_only    = up14  - (up84 | down84)
    down14_only  = down14 - (up84 | down84)
    up84_only    = up84  - (up14 | down14)
    down84_only  = down84 - (up14 | down14)
    #for fisher flipped genes
    genes_up14_down84 = up14 | down84
    genes_down14_up84 = down14 | up84
    n_up14_down84 = len(genes_up14_down84)
    n_down14_up84 = len(genes_down14_up84)
    n_up14 = len(up14)
    n_down14 = len(down14)
    n_up84 = len(up84)
    n_down84 = len(down84)
    #get jaccard
    a_up, a_down = len(up14), len(down14)
    b_up, b_down = len(up84), len(down84)
    N = len(universe)

    # Concordant use — one-sided (greater)
    k_upup = len(up14 & up84)
    k_dndn = len(down14 & down84)
    p_upup = np.nan
    p_dndn = np.nan
    if a_up>=MIN_LIST_SIZE_FOR_TESTS and b_up>=MIN_LIST_SIZE_FOR_TESTS and N>0:
        p_upup = fisher_overlap(a_up, b_up, k_upup, N, alternative="greater")
    if a_down>=MIN_LIST_SIZE_FOR_TESTS and b_down>=MIN_LIST_SIZE_FOR_TESTS and N>0:
        p_dndn = fisher_overlap(a_down, b_down, k_dndn, N, alternative="greater")
    p_concordant_combined = fisher_method_p([p_upup, p_dndn])

    # Flipped use — two-sided
    k_updn = len(up14 & down84)
    k_dnup = len(down14 & up84)
    p_updn = np.nan
    p_dnup = np.nan
    if a_up>=MIN_LIST_SIZE_FOR_TESTS and b_down>=MIN_LIST_SIZE_FOR_TESTS and N>0:
        p_updn = fisher_overlap(a_up, b_down, k_updn, N, alternative="two-sided")
    if a_down>=MIN_LIST_SIZE_FOR_TESTS and b_up>=MIN_LIST_SIZE_FOR_TESTS and N>0:
        p_dnup = fisher_overlap(a_down, b_up, k_dnup, N, alternative="two-sided")
    p_flipped_combined = fisher_method_p([p_updn, p_dnup])

    # Basic overlap (sign do not matter)
    de14 = set(merged.index[sig14]); a_all = len(de14)
    de84 = set(merged.index[sig84]); b_all = len(de84)
    k_all = len(de14 & de84) #universe=intersection
    expected_all = (a_all * b_all / N) if N else np.nan
    fold_all = (k_all / expected_all) if expected_all else np.nan
    jc_all = jaccard(de14, de84)
    fisher_all = np.nan
    if a_all>=MIN_LIST_SIZE_FOR_TESTS and b_all>=MIN_LIST_SIZE_FOR_TESTS and N>0:
        fisher_all = fisher_overlap(a_all, b_all, k_all, N, alternative="greater")

    # Concordance categorization on base set (for counts)
    def dir_from_vals(fdr, lfc):
        if (fdr < FDR_CUTOFF) and (lfc >  LFC_CUTOFF): return "Upregulated"
        if (fdr < FDR_CUTOFF) and (lfc < -LFC_CUTOFF): return "Downregulated"
        return "Non-significant"
    # define all variavles to save for files
    base["direction_14"] = [dir_from_vals(f,l) for f,l in zip(base["fdr_14"], base["logFC_14"])]
    base["direction_84"] = [dir_from_vals(f,l) for f,l in zip(base["fdr_84"], base["logFC_84"])]
    conc_mask = ((base["direction_14"]=="Upregulated") & (base["direction_84"]=="Upregulated")) | \
                ((base["direction_14"]=="Downregulated") & (base["direction_84"]=="Downregulated"))
    flip_mask = ((base["direction_14"]=="Upregulated") & (base["direction_84"]=="Downregulated")) | \
                ((base["direction_14"]=="Downregulated") & (base["direction_84"]=="Upregulated"))
    one_dir_mask = ((base["direction_14"]!="Non-significant") & (base["direction_84"]=="Non-significant")) | \
                   ((base["direction_14"]=="Non-significant") & (base["direction_84"]!="Non-significant"))
    ns_mask = (base["direction_14"]=="Non-significant") & (base["direction_84"]=="Non-significant")

    # Direction counts 
    concordant = base[conc_mask]; flipped = base[flip_mask]
    one_dir    = base[one_dir_mask]; ns = base[ns_mask]
    
    # Sign test & Spearman (use all tetseD mode)
    #spearman rho for statisctocs, binomial and spearm p for t sttaics fdr correction
    M = universe if MODE == "all_tested" else list(base.index)
    if len(M)>0:
        s14 = np.sign(d14u.loc[M, "logFC_14"].values)
        s84 = np.sign(d84u.loc[M, "logFC_84"].values)
        same = int((s14 == s84).sum())
        sign_p = binomtest(k=same, n=len(M), p=0.5, alternative="greater").pvalue
        rho, rho_p = spearmanr(d14u.loc[M, "logFC_14"].values, d84u.loc[M, "logFC_84"].values)
    else:
        same, sign_p, rho, rho_p = (0, np.nan, np.nan, np.nan)
    outdir = OUTROOT / contrast_key / celltype
    outdir.mkdir(parents=True, exist_ok=True)
    concordant.to_csv(outdir/"shared_concordant_py.csv")
    flipped.to_csv(outdir/"shared_flipped_py.csv")
    # Overlap summaries (sign not matter + sign-aware)
    ov = pd.DataFrame([{
    "CellType": celltype, "Contrast": contrast_key,
    "Universe_N": N,
    "DE_14_inter": n_DE14_inter,
    "DE_84_inter": n_DE84_inter,
    "DE_14_raw": n_DE14_raw,
    "DE_84_raw": n_DE84_raw,
    "Overlap_all": len(DE14_inter & DE84_inter),
    "Expected_all": (n_DE14_inter * n_DE84_inter / len(universe)) if len(universe) else np.nan,
    "Fold_enrichment_all": fold_all,
    "Jaccard_all": jc_all,
    "FisherP_all_enrich": fisher_all,

    "UP14": a_up, "UP84": b_up, "Overlap_UPUP": k_upup,
    "DN14": a_down, "DN84": b_down, "Overlap_DNDN": k_dndn,
    "FisherP_UPUP_greater": p_upup,
    "FisherP_DNDN_greater": p_dndn,

    "Flipped_Up14_Down84": k_updn,
    "Flipped_Down14_Up84": k_dnup,
    "Flipped_Up14_Down84_totalGenes": n_up14_down84,
    "Flipped_Down14_Up84_totalGenes": n_down14_up84,
    "Flipped_Up14_Down84_14dpiGenes": n_up14,
    "Flipped_Up14_Down84_84dpiGenes": n_down84,
    "FisherP_UPDN_twoSided": p_updn,
    "FisherP_DNUP_twoSided": p_dnup,
    "FisherP_Concordant_Combined": p_concordant_combined,
    "FisherP_Flipped_Combined": p_flipped_combined,

    "Up14_only": len(up14_only),
    "Down14_only": len(down14_only),
    "Up84_only": len(up84_only),
    "Down84_only": len(down84_only),
    "OneDir_14_total": len(up14_only) + len(down14_only),
    "OneDir_84_total": len(up84_only) + len(down84_only),

    "TotalGenes_UpUp_Test": a_up + b_up,
    "TotalGenes_DnDn_Test": a_down + b_down,
    "TotalGenes_UpDn_Test": a_up + b_down,
    "TotalGenes_DnUp_Test": a_down + b_up
    }])
    ov.to_csv(outdir/"overlap_signaware_summary_py.csv", index=False)
    # Concordance tests summary
    conc = pd.DataFrame([{
        "CellType": celltype, "Contrast": contrast_key,
        "N_for_concordance": len(M),
        "SameDirection": same, "SignTestP": sign_p,
        "SpearmanRho": rho, "SpearmanP": rho_p,
        "Concordant_ct": concordant.shape[0],
        "Flipped_ct": flipped.shape[0],
        "OneDirection_ct": one_dir.shape[0],
        "NonSigBoth_ct": ns.shape[0]
    }])

    # Plot scatter of logc of logfc of 14 and 84
    if N>0:
        common = sorted(set(d14u.index) & set(d84u.index))
        x = d14u.loc[common, "logFC_14"].values
        y = d84u.loc[common, "logFC_84"].values
        plt.figure(figsize=(5.2,5.2))
        plt.scatter(x, y, s=8, alpha=0.6)
        lims = [min(x.min(), y.min()), max(x.max(), y.max())]
        plt.plot(lims, lims, linestyle="--")
        plt.xlabel("Log2FC (14 dpi)"); plt.ylabel("Log2FC (84 dpi)")
        plt.title(f"{celltype} {contrast_key} — Spearman ρ={rho:.2f}")
        plt.tight_layout(); plt.savefig(outdir/"logFC14_vs_logFC84_scatter_py.png", dpi=300); plt.close()

    # Compact counts table
    pd.DataFrame({
        "CellType":[celltype], "Contrast":[contrast_key],
        "Concordant":[concordant.shape[0]],
        "Flipped":[flipped.shape[0]],
        "OneDirection":[one_dir.shape[0]],
        "NonSigBoth":[ns.shape[0]],
        "DE_14_all":[a_all], "DE_84_all":[b_all], "Universe":[N],
        "Jaccard_DE":[jc_all],
        "FisherP_all_enrich":[fisher_all],
        "FisherP_Concordant_Combined":[p_concordant_combined],
        "FisherP_Flipped_Combined":[p_flipped_combined]
    }).to_csv(outdir/"direction_counts_summary_py.csv", index=False)

    # Return for global summary
    return ov.assign(**conc.iloc[0].to_dict()).iloc[0].to_dict()

#main 
all_rows = []
for contrast in CONTRASTS.keys():
    for ct in CELLTYPES:
        try:
            row = analyze(ct, contrast)
            all_rows.append(row)
        except FileNotFoundError as e:
            print(f"[WARN] {e}")
        except Exception as e:
            print(f"[ERROR] {ct} {contrast}: {e}")
#combine all 
summary_df = pd.DataFrame(all_rows)
summary_df.to_csv(OUTROOT/"summary_all_celltypes_contrasts_py.csv", index=False)

# plotting
if not summary_df.empty:
    # Spearman rho heatmap by CellType x Contrast
    pivot_rho = summary_df.pivot_table(index="CellType", columns="Contrast", values="SpearmanRho", aggfunc="first")
    pivot_rho = pivot_rho.reindex(CELLTYPES)
    plt.figure(figsize=(6.5, max(3.5, 0.35*len(CELLTYPES))))
    plt.imshow(pivot_rho.values, aspect="auto")
    plt.colorbar(label="Spearman ρ (logFC 14 vs 84)")
    plt.yticks(range(len(pivot_rho.index)), pivot_rho.index)
    plt.xticks(range(len(pivot_rho.columns)), pivot_rho.columns)
    plt.title("Effect-size concordance across cell types and contrasts")
    plt.tight_layout()
    plt.savefig(OUTROOT/"spearman_rho_heatmap_celltype_by_contrast_py.png", dpi=300)
    plt.close()

    # Per-contrast bar of Spearman ρ
    for contrast in CONTRASTS.keys():
        sub = summary_df[summary_df["Contrast"]==contrast].set_index("CellType").reindex(CELLTYPES)
        plt.figure(figsize=(7.5,3.8))
        plt.bar(sub.index, sub["SpearmanRho"].values)
        plt.xticks(rotation=45, ha="right"); plt.ylim(-1,1)
        plt.ylabel("Spearman ρ"); plt.title(f"Spearman ρ (14 vs 84) — {contrast}")
        plt.tight_layout(); plt.savefig(OUTROOT/f"spearman_rho_bar_{contrast}_py.png", dpi=300); plt.close()

print("Done. Summaries & plots saved under:", OUTROOT)


Done. Summaries & plots saved under: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84


In [10]:
#saving files
cols = ["gene","fdr_14","logFC_14","direction_14","fdr_84","logFC_84","direction_84"]
for contrast in CONTRASTS.keys():
    for celltype in CELLTYPES:
        outdir = OUTROOT / contrast / celltype
        ov_path = outdir / "overlap_signaware_summary_py.csv"
        if not ov_path.exists():
            continue

        ov = pd.read_csv(ov_path)
        p_upup = float(ov["FisherP_UPUP_greater"].iloc[0]) if "FisherP_UPUP_greater" in ov else np.nan
        p_dndn = float(ov["FisherP_DNDN_greater"].iloc[0]) if "FisherP_DNDN_greater" in ov else np.nan
        p_updn = float(ov["FisherP_UPDN_twoSided"].iloc[0]) if "FisherP_UPDN_twoSided" in ov else np.nan
        p_dnup = float(ov["FisherP_DNUP_twoSided"].iloc[0]) if "FisherP_DNUP_twoSided" in ov else np.nan

        conc = outdir / "shared_concordant_py.csv"
        flip = outdir / "shared_flipped_py.csv"

        if conc.exists():
            dfc = pd.read_csv(conc)
            if p_upup < 0.05:
                upup = dfc[(dfc["direction_14"]=="Upregulated") & (dfc["direction_84"]=="Upregulated")]
                upup[cols].to_csv(outdir/"fisher_UPUP_sig_genes.csv", index=False)
            if p_dndn < 0.05:
                dndn = dfc[(dfc["direction_14"]=="Downregulated") & (dfc["direction_84"]=="Downregulated")]
                dndn[cols].to_csv(outdir/"fisher_DNDN_sig_genes.csv", index=False)

        if flip.exists():
            dff = pd.read_csv(flip)
            if p_updn < 0.05:
                updn = dff[(dff["direction_14"]=="Upregulated") & (dff["direction_84"]=="Downregulated")]
                updn[cols].to_csv(outdir/"fisher_UPDN_sig_genes.csv", index=False)
            if p_dnup < 0.05:
                dnup = dff[(dff["direction_14"]=="Downregulated") & (dff["direction_84"]=="Upregulated")]
                dnup[cols].to_csv(outdir/"fisher_DNUP_sig_genes.csv", index=False)

In [7]:
OUTROOT = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84")
SUMMARY_CSV = OUTROOT / "summary_all_celltypes_contrasts_py.csv"

CELLTYPES = ["Monocytes","NK","pDCs","ASCs","B_cells","CD2_neg_T","CD2_pos_T","CD4_Tcells","CD8T_cells"]
CONTRASTS = ["EP","EC","PC"]


MARK_SIGNIFICANCE = True
ALPHA = 0.05  

df = pd.read_csv(SUMMARY_CSV)

#define using FDR correction 
rho = df.pivot_table(index="CellType", columns="Contrast", values="SpearmanRho", aggfunc="first")
rho = rho.reindex(index=CELLTYPES, columns=CONTRASTS)


pvals = df.pivot_table(index="CellType", columns="Contrast", values="SpearmanP", aggfunc="first")
pvals = pvals.reindex(index=CELLTYPES, columns=CONTRASTS)

def bh_fdr(p):
    q = p.copy()
    for c in p.columns:
        col = p[c].values.astype(float)
        m = np.sum(~np.isnan(col))
        if m == 0:
            q[c] = np.nan
            continue
        order = np.argsort(col, kind="mergesort")
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(col)+1)
        # BH: q = p * m / rank
        qvals = col * m / ranks
        # enforce monotonicity
        qvals_sorted = qvals[order]
        for i in range(len(qvals_sorted)-2, -1, -1):
            qvals_sorted[i] = min(qvals_sorted[i], qvals_sorted[i+1])
        qvals[order] = qvals_sorted
        q[c] = qvals
    return q

qvals = bh_fdr(pvals) if MARK_SIGNIFICANCE else None


plt.figure(figsize=(6.6, max(3.6, 0.35*len(CELLTYPES))))
im = plt.imshow(rho.values, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, label="Spearman ρ (log2FC: 14 vs 84)")
plt.yticks(range(len(CELLTYPES)), CELLTYPES)
plt.xticks(range(len(CONTRASTS)), CONTRASTS)
plt.title("Effect-size concordance across cell types and treatments")


if MARK_SIGNIFICANCE and qvals is not None:
    for i, ct in enumerate(CELLTYPES):
        for j, con in enumerate(CONTRASTS):
            q = qvals.loc[ct, con]
            if pd.notna(q) and q < ALPHA:
                # draw a small  asterick near the cell center
                plt.plot(j, i, marker="*", markersize=2, markerfacecolor="none", markeredgecolor="black", linewidth=1)

plt.tight_layout()
plt.savefig(OUTROOT / "spearman_rho_heatmap_celltype_by_contrast.png", dpi=300)
plt.close()


plt.figure(figsize=(6.6, max(3.6, 0.35*len(CELLTYPES))))
im = plt.imshow(rho.values, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, label="Spearman ρ (log2FC: 14 vs 84)")
plt.yticks(range(len(CELLTYPES)), CELLTYPES)
plt.xticks(range(len(CONTRASTS)), CONTRASTS)
plt.title("Effect-size concordance (with ρ labels)")


for i in range(len(CELLTYPES)):
    for j in range(len(CONTRASTS)):
        val = rho.values[i, j]
        if np.isnan(val):
            txt = "NA"
        else:
            txt = f"{val:.2f}"
        # choose text color by value
        color = "white" if not np.isnan(val) and abs(val) > 0.5 else "black"
        plt.text(j, i, txt, ha="center", va="center", fontsize=9, color=color)

# and optional significance ring
if MARK_SIGNIFICANCE and qvals is not None:
    for i, ct in enumerate(CELLTYPES):
        for j, con in enumerate(CONTRASTS):
            q = qvals.loc[ct, con]
            if pd.notna(q) and q < ALPHA:
                plt.plot(j, i, marker="*", markersize=2, markerfacecolor="none", markeredgecolor="black", linewidth=1)

plt.tight_layout()
plt.savefig(OUTROOT / "spearman_rho_heatmap_celltype_by_contrast_with_labels.png", dpi=300)
plt.close()

print("Saved heatmaps to:", OUTROOT)

Saved heatmaps to: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84


In [2]:
#get DE volcano plots for figs#

In [15]:
# Input files
deg_files = {
    "D14": "/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/Project_Fang_10X/Project_Fang_10X/MAST_DE_14dpi/Monocytes/Mast_Mono_ext_per.csv", 
    "D84": "/work/ABG/mkapoor/mkapoor/PRRSV/PRRSV_cellranger_v97/MAST_DE_84dpi/Monocytes/Mast_Monocytes_ext_per.csv"
}
# Output folder
OUTDIR = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84/Volcano_Plots")
OUTDIR.mkdir(parents=True, exist_ok=True)
# Cutoffs
FDR_CUTOFF = 0.05
LFC_CUTOFF = 0.05  
TOP_LABELS = 30
GENE_COL = "primerid"
LFC_COL = "logFC"
FDR_COL = "fdr"
def plot_volcano(df, title, outfile):
    df = df.copy()
    df[FDR_COL] = pd.to_numeric(df[FDR_COL], errors="coerce")
    df[LFC_COL] = pd.to_numeric(df[LFC_COL], errors="coerce")
    df = df.dropna(subset=[GENE_COL, LFC_COL, FDR_COL])
    df["neglog10FDR"] = -np.log10(df[FDR_COL])
    df["significant"] = (df[FDR_COL] < FDR_CUTOFF) & (abs(df[LFC_COL]) > LFC_CUTOFF)
    sig_df = df[df["significant"]].copy()
    # Sort by FDR and take top N 
    label_df = sig_df.sort_values(FDR_COL).head(TOP_LABELS)

    plt.figure(figsize=(9, 7))
    sns.scatterplot(
        data=df, x=LFC_COL, y="neglog10FDR",
        color="gray", alpha=0.6, s=20
    )
    sns.scatterplot(
        data=sig_df, x=LFC_COL, y="neglog10FDR",
        color="red", alpha=0.8, s=30
    )

    # Add cutoffs
    plt.axhline(-np.log10(FDR_CUTOFF), color="black", linestyle="--", linewidth=1)
    plt.axvline(-LFC_CUTOFF, color="black", linestyle="--", linewidth=1)
    plt.axvline(LFC_CUTOFF, color="black", linestyle="--", linewidth=1)

    # Add labels with red boxes
    texts = []
    for _, row in label_df.iterrows():
        texts.append(
            plt.text(row[LFC_COL], row["neglog10FDR"], row[GENE_COL],
                     fontsize=9, color="black", weight="bold",
                     bbox=dict(boxstyle="round,pad=0.2", edgecolor="red", facecolor="white", alpha=0.8))
        )
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color='red', lw=0.8))

    plt.title(f"Volcano Plot: {title}", fontsize=14, weight='bold')
    plt.xlabel("Log2 Fold Change", fontsize=12)
    plt.ylabel("-Log10(FDR)", fontsize=12)

    sns.despine()
    plt.tight_layout()
    plt.savefig(outfile.with_suffix(".png"), dpi=300)
    plt.savefig(outfile.with_suffix(".pdf"))
    plt.close()
    print(f"Saved volcano plot: {outfile.with_suffix('.png')}")
#run
for timepoint, filepath in deg_files.items():
    df = pd.read_csv(filepath)
    plot_volcano(df, title=f"{timepoint} Differential Expression-EP",
                 outfile=OUTDIR / f"Volcano_{timepoint}")

Saved volcano plot: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84/Volcano_Plots/Volcano_D14.png
Saved volcano plot: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84/Volcano_Plots/Volcano_D84.png


In [18]:
#differnt colors
def plot_volcano(df, title, outfile):
    df = df.copy()

    df[FDR_COL] = pd.to_numeric(df[FDR_COL], errors="coerce")
    df[LFC_COL] = pd.to_numeric(df[LFC_COL], errors="coerce")
    df = df.dropna(subset=[GENE_COL, LFC_COL, FDR_COL])

    df["neglog10FDR"] = -np.log10(df[FDR_COL])

    df["significant"] = (df[FDR_COL] < FDR_CUTOFF) & (abs(df[LFC_COL]) > LFC_CUTOFF)

    # Assign colors:
    #   - Green = Low logFC (< -LFC_CUTOFF)
    #   - Orange = High logFC (> LFC_CUTOFF)
    #   - Gray = Not significant
    df["color"] = "gray"
    df.loc[(df["significant"]) & (df[LFC_COL] > LFC_CUTOFF), "color"] = "darkorange"
    df.loc[(df["significant"]) & (df[LFC_COL] < -LFC_CUTOFF), "color"] = "green"

    label_df = df[df["significant"]].sort_values(FDR_COL).head(TOP_LABELS)

    plt.figure(figsize=(9, 7))
    sns.scatterplot(
        data=df, x=LFC_COL, y="neglog10FDR",
        hue="color", palette={"gray":"gray", "green":"green", "darkorange":"darkorange"},
        alpha=0.7, s=30, legend=False
    )

    plt.axhline(-np.log10(FDR_CUTOFF), color="black", linestyle="--", linewidth=1)
    plt.axvline(-LFC_CUTOFF, color="black", linestyle="--", linewidth=1)
    plt.axvline(LFC_CUTOFF, color="black", linestyle="--", linewidth=1)

    texts = []
    for _, row in label_df.iterrows():
        texts.append(
            plt.text(
                row[LFC_COL], row["neglog10FDR"], row[GENE_COL],
                fontsize=9, color="black", weight="bold",
                bbox=dict(boxstyle="round,pad=0.2", edgecolor="red", facecolor="white", alpha=0.8)
            )
        )
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color='red', lw=0.8))

    plt.title(f"Volcano Plot: {title}", fontsize=14, weight='bold')
    plt.xlabel("Log2 Fold Change", fontsize=12)
    plt.ylabel("-Log10(FDR)", fontsize=12)

    sns.despine()
    plt.tight_layout()

    plt.savefig(outfile.with_suffix(".png"), dpi=300)
    plt.savefig(outfile.with_suffix(".pdf"))
    plt.close()
    print(f"Saved volcano plot: {outfile.with_suffix('.png')}")

# Run 
for timepoint, filepath in deg_files.items():
    df = pd.read_csv(filepath)
    plot_volcano(df, title=f"{timepoint} Differential Expression - EP",
                 outfile=OUTDIR / f"Volcano_{timepoint}")

Saved volcano plot: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84/Volcano_Plots/Volcano_D14.png
Saved volcano plot: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84/Volcano_Plots/Volcano_D84.png
